## Configuration


In [ ]:
dbutils.widgets.text("catalog", "training_lab")
dbutils.widgets.text("schema", "student_01")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", schema)
ns = f"{catalog}.{schema}"
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")
print("Working namespace:", ns)
from pyspark.sql import functions as F


## Customer clustering


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
rows=[(i,float(1+i%3),float(100+20*(i%4))) for i in range(10)]
rows += [(i,float(12+i%3),float(1500+100*(i%4))) for i in range(10,20)]
customers=spark.createDataFrame(rows,"customer_id int, visits double, spend double")
pipeline=Pipeline(stages=[
 VectorAssembler(inputCols=["visits","spend"],outputCol="raw_features"),
 StandardScaler(inputCol="raw_features",outputCol="features",withStd=True,withMean=True),
 KMeans(k=2,seed=18,featuresCol="features",predictionCol="cluster")])
model=pipeline.fit(customers)
segments=model.transform(customers)
display(segments.groupBy("cluster").agg(F.count("*").alias("customers"),
 F.avg("visits").alias("avg_visits"),F.avg("spend").alias("avg_spend")))
print("Silhouette:",ClusteringEvaluator(featuresCol="features",
 predictionCol="cluster").evaluate(segments))
# Descriptive clustering of these 20 rows. No claim of unseen-data accuracy.
# Cluster IDs 0 and 1 can swap. Interpret profiles, not numeric IDs.
